<b> Title: </b> Understanding Performance Dynamics in Modern Formula 1

<b> Topic: </b> Analysing the relationship between qualifying performance and race outcomes, intra-team competitive dynamics, and strategic tyre management using data from 2018 to 2024 from the datasource FastF1. 

<b> Research Question 1: </b>  Qualifying-Race Performance Correlation

<u> “What is the relationship between qualifying session performance metrics and final race standings across different circuits, seasons, and regulatory eras?” </u>

<u> <i> <center> 1a) Does the qualifying-race relationship change depending on the context of circuit type and overtaking difficulty? </center> </i> </u>

<u> <i> <center> 1b) Do the regulations affecting overtaking introduced in 2022 affect the qualifying-race correlation? </center> </i> </u>


# Current Plan

1. intro, title, topic, question, datasource in this case, my questions and outcomes. results summary on top?
2. eda, setting up environmetn, turning offf warnings, importing pacakges, downloading and callng fastf1 data.
3. constructing dataframe. years 2018,2024. for multinomial model

define model specific things later like cv folds. traint est etc. shoul dbe within the specific model section. 

More detailed in my doc.

# ST312 Notebook Plan (FastF1) — Complete Step-by-Step
0. Notebook purpose and “what this notebook produces”

1. Environment + reproducibility setup

2. Imports and project configuration

3. Data extraction plan (what you will pull from FastF1)

4. Build the event-driver dataset (core dataset for multinomial regression)
4.1 Get the race calendar properly (no guessing rounds)
4.2 Loop over events and extract session results
4.3 Merge and label
4.4 Define the multinomial outcome variable
4.5 Data quality checks


5. EDA for Question 1 (short, high-signal)
5.1 Dataset overview
5.2 Relationship plots (must-have)

6. Analysis 1 — Multinomial logistic regression with elastic net (ST310 style)
6.1 Define modelling objective and variables
6.2 Train/test split
6.3 Preprocessing pipeline
6.4 Model and hyperparameter tuning
6.5 Evaluate on test set (mandatory outputs)
6.6 Interpret results for Q1a/Q1b

7. Analysis 2 — Within-race time series / dependence using lap-by-lap position
7.1 Build lap-level panel dataset
7.2 Define dependence measures (what exactly you compute)
7.3 Compare dependence by circuit type (Q1a)
7.4 Compare dependence by era (Q1b)
7.5 Add “position change” dynamics (very interpretable)

8. Robustness checks (small but high-value)

9. Final results section (answers Q1a + Q1b explicitly)

10. Reproducibility and saving outputs

11. Notebook finishing touches (to score marks)


# Casual to be deleted 
1. multinomial regression (with elastic net using st310 stuff)
2. time series analysis within race using starting position as quaifying result and autocorrelaiton between each lap and position

Research Question 1: Qualifying-Race Performance Correlation
We will use ordinal logistic regression with a multilevel structure to examine the relationship between qualifying position and race finishing position. This approach makes sense because race positions are ordered categories (1st, 2nd, 3rd, etc.) rather than continuous numbers. We will include random effects for drivers and teams to account for the fact that we have multiple observations from the same drivers and teams over the seasons.
We will also calculate Spearman rank correlations broken down by year, circuit type, and grid position. This will let us see if the qualifying-race relationship changes depending on context. For circuit types, we'll group tracks into street circuits (like Monaco and Singapore), high-speed circuits (like Monza and Spa), and technical circuits (like Barcelona and Suzuka) to test whether overtaking difficulty affects the correlation.
If our hypothesis is correct, we expect strong correlations of at least 0.70 overall, but with differences between circuits. For example, Monaco should show stronger correlations (around 0.85) because overtaking is nearly impossible there, while Monza might be weaker (around 0.65) because the long straights make it easier to pass. We'll also split the data into 2018-2021 and 2022-2024 periods to see if the 2022 regulation changes improved racing. If the new rules worked as intended, the qualifying-race correlation should be weaker in 2022-2024 because overtaking became easier.

// maybe time series correlation within race. using starting positions as the qualifying outcome and then using st326 code for autocorrelation analysis for each laptime and position?

### Importing data and packages
Below, I import the main Python libraries needed for this project. I use FastF1 to download official timing and telemetry data for Formula 1 sessions (qualifying and race). I also import pandas/numpy for organising the data into tables, and scikit-learn for building a multinomial logistic regression model with an elastic net penalty. Elastic net combines L1 (lasso) and L2 (ridge) regularisation, which is useful when predictors are correlated and when we want a model that is both stable and selective.

In [3]:
# !pip install fastf1

  Obtaining dependency information for fastf1 from https://files.pythonhosted.org/packages/28/46/d83e255fe78c4a14d7b9fce8fb61b44be66be5be0cb9f37b27f1aeaf68d3/fastf1-3.7.0-py3-none-any.whl.metadata
  Obtaining dependency information for rapidfuzz from https://files.pythonhosted.org/packages/76/25/5b0a33ad3332ee1213068c66f7c14e9e221be90bab434f0cb4defa9d6660/rapidfuzz-3.14.3-cp311-cp311-macosx_10_9_x86_64.whl.metadata
  Obtaining dependency information for requests-cache>=1.0.0 from https://files.pythonhosted.org/packages/68/3f/dfa42bb16be96d53351aa151cb1e39fcaafe6cda01389c530a2ec809ef8a/requests_cache-1.3.0-py3-none-any.whl.metadata
  Obtaining dependency information for signalrcore from https://files.pythonhosted.org/packages/f3/16/96f9dc0c60e51b9cbbe2da0f8ea37782fd16746274e28b713b78575eaf1d/signalrcore-0.9.71-py3-none-any.whl.metadata
  Obtaining dependency information for timple>=0.1.6 from https://files.pythonhosted.org/packages/75/45/c73f9af9a9d50b0ae972d185ef2255c62524d1aa20a76531d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 5.2 MB/s eta 0:00:00
  Created wheel for msgpack: filename=msgpack-1.0.2-cp311-cp311-macosx_10_9_x86_64.whl size=15825 sha256=805a3b23ed39b0830851b0f62d1bfc0b0b93a94c2c670f733ee4a09c2045f6b6
  Stored in directory: /Users/minaheelkhan/Library/Caches/pip/wheels/17/f3/65/51600c2eee827e34dc01b7d903670e2ed4d62815b80f60f112
Successfully built msgpack
  Attempting uninstall: msgpack
    Found existing installation: msgpack 1.0.3
    Uninstalling msgpack-1.0.3:
      Successfully uninstalled msgpack-1.0.3
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.7.1
    Uninstalling typing_extensions-4.7.1:
      Successfully uninstalled typing_extensions-4.7.1
  Attempting uninstall: attrs
    Found existing installation: attrs 22.1.0
    Uninstalling attrs-22.1.0:
      Successfully uninstalled attrs-22.1.0


In [17]:
# Core data handling
import numpy as np
import pandas as pd

# FastF1: session data, laps, results
import fastf1
from fastf1 import plotting
fastf1.set_log_level("ERROR")


# Modelling + preprocessing (similar structure to your ST310 multinomial pipeline)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import matplotlib.pyplot as plt

# ignoring warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

### Pull qualifying and race results from Fast F1

Before modelling, I decide what each row in my dataset represents. For Question 1, the cleanest unit is a single driver in a single race weekend, because qualifying and race outcomes are naturally defined per driver per event. So each row will be:
(season, round, circuit, driver) → qualifying performance → race outcome.

Below, I loop through seasons and rounds and download two sessions for each event: Qualifying (“Q”) and Race (“R”). From qualifying I extract a qualifying position (or a time-gap feature), and from the race I extract finishing position and status. I then merge them into one table so that I can study how qualifying relates to race outcomes.

In [18]:
def get_event_table(year: int, round_no: int) -> pd.DataFrame:
    """
    Returns one row per driver for a given (year, round).
    Columns include qualifying info + race result info.
    """
    # --- Qualifying session ---
    q = fastf1.get_session(year, round_no, "Q")
    q.load()
    q_res = q.results.copy()

    # Keep key fields (names can vary slightly across seasons; adjust if needed)
    q_keep = q_res[["DriverNumber", "Abbreviation", "FullName", "TeamName", "Position", "Q1", "Q2", "Q3"]].copy()
    q_keep = q_keep.rename(columns={"Position": "QualiPos"})

    # Create a simple qualifying performance metric:
    # best available qualifying time (Q3 if exists else Q2 else Q1)
    q_keep["QualiTime"] = q_keep["Q3"].fillna(q_keep["Q2"]).fillna(q_keep["Q1"])

    # --- Race session ---
    r = fastf1.get_session(year, round_no, "R")
    r.load()
    r_res = r.results.copy()

    r_keep = r_res[["DriverNumber", "Abbreviation", "FullName", "TeamName", "Position", "Status", "GridPosition"]].copy()
    r_keep = r_keep.rename(columns={"Position": "RacePos", "GridPosition": "StartPos"})

    # --- Merge ---
    df = pd.merge(q_keep, r_keep, on=["DriverNumber", "Abbreviation", "FullName", "TeamName"], how="inner")

    # Add event metadata
    df["Year"] = year
    df["Round"] = round_no
    df["EventName"] = q.event["EventName"]
    df["Country"] = q.event.get("Country", np.nan)
    df["Location"] = q.event.get("Location", np.nan)

    return df

# Example: get one event
example = get_event_table(2024, 1)
example.head()

,DriverNumber,Abbreviation,FullName,TeamName,QualiPos,Q1,Q2,Q3,QualiTime,RacePos,Status,StartPos,Year,Round,EventName,Country,Location
0,1,VER,Max Verstappen,Red Bull Racing,1.0,0 days 00:01:30.031000,0 days 00:01:29.374000,0 days 00:01:29.179000,0 days 00:01:29.179000,1.0,Finished,1.0,2024,1,Bahrain Grand Prix,Bahrain,Sakhir
1,16,LEC,Charles Leclerc,Ferrari,2.0,0 days 00:01:30.243000,0 days 00:01:29.165000,0 days 00:01:29.407000,0 days 00:01:29.407000,4.0,Finished,2.0,2024,1,Bahrain Grand Prix,Bahrain,Sakhir
2,63,RUS,George Russell,Mercedes,3.0,0 days 00:01:30.350000,0 days 00:01:29.922000,0 days 00:01:29.485000,0 days 00:01:29.485000,5.0,Finished,3.0,2024,1,Bahrain Grand Prix,Bahrain,Sakhir
3,55,SAI,Carlos Sainz,Ferrari,4.0,0 days 00:01:29.909000,0 days 00:01:29.573000,0 days 00:01:29.507000,0 days 00:01:29.507000,3.0,Finished,4.0,2024,1,Bahrain Grand Prix,Bahrain,Sakhir
4,11,PER,Sergio Perez,Red Bull Racing,5.0,0 days 00:01:30.221000,0 days 00:01:29.932000,0 days 00:01:29.537000,0 days 00:01:29.537000,2.0,Finished,5.0,2024,1,Bahrain Grand Prix,Bahrain,Sakhir


### Clean + define your outcome (multinomial target)

Now I define the response variable for the multinomial regression. Instead of modelling raw finishing position as 1–20, I create categories that reflect meaningful racing outcomes. This makes interpretation easier and avoids pretending that the gap between P1 and P2 is the same as between P19 and P20.

A good default is:

Podium (1–3)

Points (4–10)

NoPoints (11–20)
Optionally: DNF as a separate category if you want.

In [8]:
def make_outcome(race_pos, status):
    # Treat DNFs as separate if status indicates retirement
    if isinstance(status, str) and ("Retired" in status or "DNF" in status or "Accident" in status):
        return "DNF"
    if pd.isna(race_pos):
        return "DNF"
    if race_pos <= 3:
        return "Podium"
    elif race_pos <= 10:
        return "Points"
    else:
        return "NoPoints"

example["Outcome"] = example.apply(lambda row: make_outcome(row["RacePos"], row["Status"]), axis=1)
example[["FullName","QualiPos","StartPos","RacePos","Status","Outcome"]].head(10)


,FullName,QualiPos,StartPos,RacePos,Status,Outcome
0,Max Verstappen,1.0,1.0,1.0,Finished,Podium
1,Charles Leclerc,2.0,2.0,4.0,Finished,Points
2,George Russell,3.0,3.0,5.0,Finished,Points
3,Carlos Sainz,4.0,4.0,3.0,Finished,Podium
4,Sergio Perez,5.0,5.0,2.0,Finished,Podium
5,Fernando Alonso,6.0,6.0,9.0,Finished,Points
6,Lando Norris,7.0,7.0,6.0,Finished,Points
7,Oscar Piastri,8.0,8.0,8.0,Finished,Points
8,Lewis Hamilton,9.0,9.0,7.0,Finished,Points
9,Nico Hulkenberg,10.0,10.0,16.0,Lapped,NoPoints


### preliminary EDA
Before fitting any regression, I do a few descriptive checks to understand the dataset: the distribution of outcomes, how often qualifying translates into points/podium, and whether the relationship looks different across seasons or circuit types. This mirrors the tennis report style, where they first show baseline win-rates and breakdowns before modelling

In [9]:
# Example EDA: outcome counts
example["Outcome"].value_counts(normalize=True)

# Quick look: average race position by qualifying position (rough diagnostic)
example.groupby("QualiPos")["RacePos"].mean().head(10)

QualiPos
1.0      1.0
2.0      4.0
3.0      5.0
4.0      3.0
5.0      2.0
6.0      9.0
7.0      6.0
8.0      8.0
9.0      7.0
10.0    16.0
Name: RacePos, dtype: float64

# Multinomial Regression. can use further code and explanaitons from my st310 coursework

Now I fit a multinomial logistic regression where the model predicts the probability of each race outcome category (Podium / Points / NoPoints / DNF). I use an elastic net penalty because many performance variables are correlated (e.g., qualifying position, grid position, qualifying times, team strength proxies). Elastic net balances ridge (stability under collinearity) and lasso (some variable selection), which is helpful when we have many overlapping predictors.

Below, I split the dataset into training and test sets. I use stratification so the outcome categories stay balanced across the split. Then I build a preprocessing pipeline: numerical variables are imputed and scaled, and categorical variables are imputed and one-hot encoded.

In [11]:
# Suppose full_df is your combined data across all events
full_df = pd.concat([get_event_table(y, r) for y in range(2018, 2025) for r in rounds], ignore_index=True)

TARGET = "Outcome"
DROP = ["FullName", "DriverNumber"]  # keep abbreviations/teams if you want

X = full_df.drop(columns=[TARGET] + [c for c in DROP if c in full_df.columns])
y = full_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop"
)


NameError: name 'rounds' is not defined

Now I tune the strength of regularisation (C) and the elastic net mixing parameter (l1_ratio).

Larger C = less regularisation

l1_ratio close to 1 behaves more like lasso

l1_ratio close to 0 behaves more like ridge

I choose the best model using cross-validated macro-F1 so each outcome category matters, not just the most common one.

In [12]:
base_model = LogisticRegression(
    multi_class="multinomial",
    solver="saga",              # required for elastic net
    penalty="elasticnet",
    max_iter=10000,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", base_model)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "model__C": np.logspace(-3, 2, 10),
    "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]   # tune the ridge/lasso mix
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    refit=True,
    verbose=1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print("Best params:", grid.best_params_)
print("Best CV macro-F1:", grid.best_score_)


NameError: name 'preprocess' is not defined

## Evaluate on Test set
Finally, I evaluate the fitted model on the held-out test set. I report accuracy, macro-F1, and a confusion matrix. Accuracy is intuitive but can be misleading when classes are imbalanced, so macro-F1 is included to treat each outcome more equally.

In [13]:
y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test Macro-F1:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


NameError: name 'best_model' is not defined

### Part 4 — Time series analysis within race (lap-by-lap dependence)

This is a great “second analysis” as long as you present it as dependence + race dynamics, not “forecasting”.

4.1 Build lap-level panel data

Now I move from race-level outcomes to lap-level dynamics. For a given race, FastF1 provides one row per driver per lap, including lap time and position. I build a “panel” dataset where each observation is:
(driver, lap) → position at lap t
and I keep the starting position (grid/qualifying) as a fixed baseline for that driver in that race.

In [14]:
def get_race_laps(year: int, round_no: int) -> pd.DataFrame:
    r = fastf1.get_session(year, round_no, "R")
    r.load()
    laps = r.laps.copy()

    # Keep what we need
    laps = laps[["Driver", "LapNumber", "Position", "LapTime"]].copy()

    # Add starting grid position from results
    res = r.results[["Abbreviation", "GridPosition"]].copy()
    res = res.rename(columns={"Abbreviation":"Driver", "GridPosition":"StartPos"})
    laps = laps.merge(res, on="Driver", how="left")

    laps["Year"] = year
    laps["Round"] = round_no
    laps["EventName"] = r.event["EventName"]

    # Convert LapTime to seconds (easier for plots)
    laps["LapTimeSeconds"] = laps["LapTime"].dt.total_seconds()

    return laps

laps_example = get_race_laps(2024, 1)
laps_example.head()


,Driver,LapNumber,Position,LapTime,StartPos,Year,Round,EventName,LapTimeSeconds
0,VER,1.0,1.0,0 days 00:01:37.284000,1.0,2024,1,Bahrain Grand Prix,97.284
1,VER,2.0,1.0,0 days 00:01:36.296000,1.0,2024,1,Bahrain Grand Prix,96.296
2,VER,3.0,1.0,0 days 00:01:36.753000,1.0,2024,1,Bahrain Grand Prix,96.753
3,VER,4.0,1.0,0 days 00:01:36.647000,1.0,2024,1,Bahrain Grand Prix,96.647
4,VER,5.0,1.0,0 days 00:01:37.173000,1.0,2024,1,Bahrain Grand Prix,97.173


### 4.2 Autocorrelation of position within a race

Next, I measure how “sticky” position is inside a race. If overtaking is hard, then a driver’s position at lap t should be highly correlated with their position at lap t+1, t+2, etc. I compute the autocorrelation function (ACF) of position for each driver (or aggregated across drivers) and compare these patterns across circuit types and eras (pre/post 2022).

In [15]:
def acf(x, max_lag=10):
    x = pd.Series(x).dropna().values
    x = x - np.mean(x)
    denom = np.sum(x**2)
    out = []
    for k in range(max_lag + 1):
        num = np.sum(x[k:] * x[:len(x)-k])
        out.append(num / denom if denom > 0 else np.nan)
    return np.array(out)

# Example: one driver in one race
one_driver = laps_example[laps_example["Driver"] == laps_example["Driver"].iloc[0]].sort_values("LapNumber")
acf_vals = acf(one_driver["Position"], max_lag=10)
acf_vals


array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan])

4.3 “Starting position vs race trajectory”

Finally, I connect qualifying (starting position) to within-race movement. A simple metric is position change by lap:

Δ𝑖,𝑡=Position
𝑖
,
𝑡
−
StartPos
𝑖
Δ
i,t
	​

=Position
i,t
	​

−StartPos
i
	​


If qualifying matters a lot, most drivers will have trajectories close to 0 (little movement). If overtaking is easier, we should see larger deviations.

In [16]:
laps_example["PosChange"] = laps_example["Position"] - laps_example["StartPos"]

# Average absolute movement per lap (race-level “overtaking activity” proxy)
overtake_proxy = laps_example.groupby("LapNumber")["PosChange"].apply(lambda s: np.nanmean(np.abs(s)))
overtake_proxy.head()


LapNumber
1.0    2.0
2.0    2.1
3.0    2.3
4.0    2.3
5.0    2.4
Name: PosChange, dtype: float64